In [ ]:
%load_ext autoreload
%autoreload 2

# Network Analysis

This notebook replicates the **Network Analysis** panel that was removed from the Dash webapp for performance reasons.
All interactive fields are plain Python variables — change them in the **Configuration** cell below, then re-run the cells you need.

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on sys.path so we can import from visualize_webapp.
_PROJECT_ROOT = Path(".").resolve()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display

pio.renderers.default = "notebook"
# White backgrounds (app builders use transparent _style(); dark notebook themes need this)
pio.templates.default = "plotly_white"
import visualize_webapp.app as _vwa

# Save the real _style only once — re-running this cell must not alias the wrapper to itself.
if not hasattr(_vwa, "_style_original"):
    _vwa._style_original = _vwa._style


def _style_white_bg(fig, **kw):
    _vwa._style_original(fig, **kw)
    fig.update_layout(paper_bgcolor="white", plot_bgcolor="white")
    return fig


_vwa._style = _style_white_bg

from visualize_webapp.app import (
    scan_results,
    _metrics,
    _pp_neuron_digit,
    _pp_dead,
    _pp_train_act,
    _compact_neuron_label,
    per_neuron_recovery_counts,
    _build_layer_inactive_count,
    _build_layer_digit_count_plot,
    _build_layer_digit_count_non_head_combined,
    _build_recovery_cumsum_plot,
    _build_dead_layer_traces,
    _build_dead_neuron_pre_nl_grid,
    _build_dead_neuron_inactivity_bar,
)
from visualize_webapp.notebook.network_analysis_mpl import (
    mpl_dead_layer_traces,
    mpl_layer_digit_count_non_head_combined,
    mpl_layer_digit_count_plot,
    mpl_layer_inactive_count,
    mpl_recovery_cumsum,
    split_ever_dead_by_final_checkpoint,
)

## Configuration

Set the four identifiers below to select which run to analyse.
Run the **Discovery** cell first if you are unsure of the exact string values.

## Discovery

Run this cell to see every available experiment / model / run / optimizer combination.

In [ ]:
tree = scan_results()
for eid, models in sorted(tree.items()):
    for mid, runs in sorted(models.items()):
        for rid, oids in sorted(runs.items()):
            for oid in oids:
                print(f"  experiment={eid!r}  model={mid!r}  run={rid!r}  optimizer={oid!r}")

In [ ]:
# __ CONFIG ____________________________________________________________
MODEL_ID      = "dnn_5x64"
EXPERIMENT_ID = "pretrain_relabelC_0_1_2_pt300_st100"
RUN_ID        = "66cd1c"
OPTIMIZER_ID  = "pure_shampoo_lr0.01"

USE_BOXPLOT = False
# When True, sections above "Dead neuron detail" use matplotlib instead of Plotly.
USE_MATPLOTLIB = True

# ____________________________________________________________________________

## Data loading

In [ ]:
df_nd   = _pp_neuron_digit(EXPERIMENT_ID, MODEL_ID, OPTIMIZER_ID, rid=RUN_ID)
df_dead = _pp_dead(EXPERIMENT_ID, MODEL_ID, OPTIMIZER_ID, rid=RUN_ID)
metrics = _metrics(EXPERIMENT_ID, MODEL_ID, OPTIMIZER_ID, rid=RUN_ID)
pp_ta   = _pp_train_act(EXPERIMENT_ID, MODEL_ID, OPTIMIZER_ID, rid=RUN_ID)

if df_nd is None or df_nd.empty:
    print("WARNING: post_processing_neuron_digit.csv not found for this run.")
    print("Re-run training to generate post-processing data.")
else:
    layer_order = list(dict.fromkeys(df_nd["layer_name"].tolist()))
    print(f"Layers: {layer_order}")

if df_dead is None or df_dead.empty:
    ever_dead_nids = []
    dead_at_end_nids = set()
    recovered_dead_nids = set()
    print("No dead-neuron data found.")
else:
    ever_dead_nids = sorted(df_dead[df_dead["is_dead"]]["neuron_id"].unique().tolist())
    dead_at_end_nids, recovered_dead_nids = split_ever_dead_by_final_checkpoint(df_dead)
    print(f"Dead neurons ({len(ever_dead_nids)}): {ever_dead_nids}")
    print(
        f"  Still dead at last checkpoint (perpetually dead at end): {len(dead_at_end_nids)}"
    )
    print(f"  Recovered dead (alive at last checkpoint): {len(recovered_dead_nids)}")

## Inactive neuron count per digit

One line chart per layer: how many neurons are inactive for each digit across checkpoints.

In [ ]:
if df_nd is not None and not df_nd.empty:
    for layer_name in layer_order:
        if USE_MATPLOTLIB:
            fig = mpl_layer_inactive_count(layer_name, df_nd, metrics)
            display(fig)
            plt.close(fig)
        else:
            fig = _build_layer_inactive_count(layer_name, df_nd, metrics)
            fig.show()
else:
    print("No data — skipped.")

## Assigned digit count distribution

Boxplot per layer: distribution of the number of digits a neuron is *assigned* to, across checkpoints.

In [ ]:
if df_nd is not None and not df_nd.empty:
    for layer_name in layer_order:
        if USE_MATPLOTLIB:
            fig = mpl_layer_digit_count_plot(
                layer_name, df_nd, metrics, "assigned", "Assigned", boxplot=USE_BOXPLOT
            )
            display(fig)
            plt.close(fig)
        else:
            fig = _build_layer_digit_count_plot(
                layer_name, df_nd, metrics, "assigned", "Assigned", boxplot=USE_BOXPLOT
            )
            fig.show()
    if not USE_BOXPLOT:
        if USE_MATPLOTLIB:
            fig = mpl_layer_digit_count_non_head_combined(
                df_nd, metrics, layer_order, "assigned", "Assigned"
            )
            display(fig)
            plt.close(fig)
        else:
            fig = _build_layer_digit_count_non_head_combined(
                df_nd, metrics, layer_order, "assigned", "Assigned"
            )
            fig.show()
else:
    print("No data — skipped.")

## Inactive digit count distribution

Boxplot per layer: distribution of the number of digits a neuron is *inactive* for, across checkpoints.

In [ ]:
if df_nd is not None and not df_nd.empty:
    for layer_name in layer_order:
        if USE_MATPLOTLIB:
            fig = mpl_layer_digit_count_plot(
                layer_name, df_nd, metrics, "inactive", "Inactive", boxplot=USE_BOXPLOT
            )
            display(fig)
            plt.close(fig)
        else:
            fig = _build_layer_digit_count_plot(
                layer_name, df_nd, metrics, "inactive", "Inactive", boxplot=USE_BOXPLOT
            )
            fig.show()
    if not USE_BOXPLOT:
        if USE_MATPLOTLIB:
            fig = mpl_layer_digit_count_non_head_combined(
                df_nd, metrics, layer_order, "inactive", "Inactive"
            )
            display(fig)
            plt.close(fig)
        else:
            fig = _build_layer_digit_count_non_head_combined(
                df_nd, metrics, layer_order, "inactive", "Inactive"
            )
            fig.show()
else:
    print("No data — skipped.")

## Dead neuron traces

For each layer: one trace per neuron — inactive-digit count over time.

- **Perpetually dead** (still dead at the **last** checkpoint): matches `is_ppd` at the end of training when that column exists.
- **Recovered dead**: were `is_dead` at some earlier checkpoint but **not** dead at the last checkpoint.

### Perpetually dead (still dead at last checkpoint)

In [ ]:
if df_dead is not None and not df_dead.empty and len(dead_at_end_nids) > 0:
    for layer_name in layer_order:
        layer_has = (
            (df_dead["layer_name"] == layer_name)
            & df_dead["is_dead"]
            & df_dead["neuron_id"].astype(str).isin(dead_at_end_nids)
        ).any()
        if layer_has:
            if USE_MATPLOTLIB:
                fig = mpl_dead_layer_traces(
                    layer_name,
                    df_nd,
                    df_dead,
                    metrics,
                    neuron_ids=dead_at_end_nids,
                    title="Perpetually dead neurons",
                    empty_message="No perpetually dead neurons in this layer.",
                )
                display(fig)
                plt.close(fig)
            else:
                fig = _build_dead_layer_traces(
                    layer_name,
                    df_nd,
                    df_dead,
                    metrics,
                    neuron_ids=dead_at_end_nids,
                    title="Perpetually dead neurons",
                    empty_message="No perpetually dead neurons in this layer.",
                )
                fig.show()
else:
    print("No perpetually dead neurons — skipped.")

### Recovered dead (alive at last checkpoint)

In [ ]:
if df_dead is not None and not df_dead.empty and len(recovered_dead_nids) > 0:
    for layer_name in layer_order:
        layer_has = (
            (df_dead["layer_name"] == layer_name)
            & df_dead["is_dead"]
            & df_dead["neuron_id"].astype(str).isin(recovered_dead_nids)
        ).any()
        if layer_has:
            if USE_MATPLOTLIB:
                fig = mpl_dead_layer_traces(
                    layer_name,
                    df_nd,
                    df_dead,
                    metrics,
                    neuron_ids=recovered_dead_nids,
                    title="Recovered dead neurons",
                    empty_message="No recovered dead neurons in this layer.",
                )
                display(fig)
                plt.close(fig)
            else:
                fig = _build_dead_layer_traces(
                    layer_name,
                    df_nd,
                    df_dead,
                    metrics,
                    neuron_ids=recovered_dead_nids,
                    title="Recovered dead neurons",
                    empty_message="No recovered dead neurons in this layer.",
                )
                fig.show()
else:
    print("No recovered dead neurons — skipped.")

## Dead neuron detail

Set `DEAD_NEURON_ID` in the **Configuration** cell to one of the full IDs from the **Data loading** cell (or from the list below). Plot titles and legends use the compact form `<layer>:<index>` (e.g. `h0:16`).

Available neurons printed below for convenience:

In [ ]:
print("Available dead neuron IDs (compact label — use full string for DEAD_NEURON_ID):")
for nid in ever_dead_nids:
    print(f"  {_compact_neuron_label(nid):12}  {nid!r}")

### Pre-NL activation grid

2×5 subplots — mean pre-nonlinearity activation over training iterations, one panel per digit.
Y-axis is forced to end at 0 to highlight how the neuron dies.

In [ ]:
# Config 2 


# Set to the full neuron_id string (e.g. "dnn|hidden.0|neuron|42"); else None to skip
DEAD_NEURON_ID = "dnn|hidden.6|neuron|33"# "dnn|hidden.2|neuron|6" # "dnn|hidden.2|neuron|41"

In [ ]:
if DEAD_NEURON_ID is not None:
    fig = _build_dead_neuron_pre_nl_grid(
        DEAD_NEURON_ID, EXPERIMENT_ID, MODEL_ID, OPTIMIZER_ID, rid=RUN_ID
    )
    fig.show()
else:
    print("DEAD_NEURON_ID is None — set it in the Configuration cell to render this plot.")

### Inactivity ratio on full training set

Bar chart: fraction of training samples that produce zero post-ReLU activation, per digit.

In [ ]:
if DEAD_NEURON_ID is not None:
    if pp_ta:
        fig = _build_dead_neuron_inactivity_bar(DEAD_NEURON_ID, pp_ta)
        fig.show()
    else:
        print("post_processing_train_act.npz not found for this run — bar chart unavailable.")
else:
    print("DEAD_NEURON_ID is None — set it in the Configuration cell to render this plot.")

## Cumulative recoveries (hidden layers)

One line per hidden layer: cumulative count of **recovery** events (neuron was `is_dead` at checkpoint *i* and not dead at *i+1*). The output layer (`head`) is omitted. Values are recomputed from `df_dead` when you run the cell.

In [ ]:
if df_dead is not None and not df_dead.empty:
    if USE_MATPLOTLIB:
        fig = mpl_recovery_cumsum(df_dead, metrics, layer_order)
        plt.show()
    else:
        fig = _build_recovery_cumsum_plot(df_dead, metrics, layer_order)
        fig.show()
else:
    print("No dead-neuron table — skipped.")

In [ ]:
# Hidden-layer neurons that were ever dead: those with at most one dead→alive transition.
_rc = per_neuron_recovery_counts(df_dead, layer_order) if df_dead is not None else {}
if df_dead is None or df_dead.empty:
    ever_dead_hidden: set[str] = set()
else:
    hid = df_dead["layer_name"].astype(str) != "head"
    dm = pd.Series(df_dead["is_dead"]).fillna(False)
    if dm.dtype == object:
        dead_m = dm.astype(str).str.lower().isin(("true", "1", "t"))
    else:
        dead_m = dm.astype(bool)
    ever_dead_hidden = {
        str(x)
        for x in df_dead.loc[hid & dead_m, "neuron_id"].tolist()
    }
_at_most_1 = sorted(
    (nid, c) for nid, c in _rc.items() if c <= 1 and nid in ever_dead_hidden
)
print(
    f"Hidden neurons that were dead at some point and have <=1 recovery ({len(_at_most_1)} of {len(ever_dead_hidden)} dead hidden):"
)
for nid, c in _at_most_1:
    print(f"  {c} recovery/recoveries: {_compact_neuron_label(nid)} — {nid}")

In [ ]:
ppd = pd.Series(df_dead["is_ppd"]).fillna(False)
ppd_m = ppd.astype(str).str.lower().isin(("true", "1", "t"))
ppd__hidden = {
        str(x) for x in df_dead.loc[hid & ppd_m, "neuron_id"].tolist()
}
_at_most_1_ppd = sorted(
    (nid, c) for nid, c in _rc.items() if c <= 1 and nid in ppd__hidden
)
print(
    f"Hidden neurons that are perpetually dead <=1 recovery ({len(_at_most_1_ppd)} of {len(ppd__hidden)} perpetually dead):"
)
for nid, c in _at_most_1_ppd:
    print(f"  {c} recovery/recoveries: {_compact_neuron_label(nid)} — {nid}")

In [ ]:
# Neurons which are *not* PPD and have exactly 1 recovery

# Calculate set of hidden neurons which are NOT perpetually dead (not PPD)
not_ppd__hidden = ever_dead_hidden - ppd__hidden

# Find those with exactly 1 recovery
exactly_1_non_ppd = sorted(
    (nid, c) for nid, c in _rc.items() if c == 1 and nid in not_ppd__hidden
)

print(
    f"Hidden neurons that are NOT perpetually dead and have exactly 1 recovery ({len(exactly_1_non_ppd)} of {len(not_ppd__hidden)} non-PPD hidden):"
)
for nid, c in exactly_1_non_ppd:
    print(f"  {c} recovery: {_compact_neuron_label(nid)} — {nid}")

In [ ]:
from visualize_webapp.app import _NETWORK_SAMPLES_PER_DIGIT

if df_nd is None or df_nd.empty or df_dead is None or df_dead.empty:
    reassigned_neuron_ids = []
    df_reassigned_summary = pd.DataFrame()
    print("No neuron-digit or dead-neuron data — skipped.")
else:
    first_dead = (
        df_dead.loc[df_dead["is_dead"], ["neuron_id", "checkpoint_idx"]]
        .groupby("neuron_id", sort=False)["checkpoint_idx"]
        .min()
        .rename("first_dead_checkpoint")
    )
    assigned = df_nd.loc[
        df_nd["status"] == "assigned",
        ["neuron_id", "layer_name", "checkpoint_idx", "digit"],
    ]
    merged = assigned.merge(first_dead.reset_index(), on="neuron_id", how="inner")
    after_dead = merged[merged["checkpoint_idx"] > merged["first_dead_checkpoint"]]

    reassigned_neuron_ids = sorted(after_dead["neuron_id"].unique().tolist())

    if after_dead.empty:
        df_reassigned_summary = pd.DataFrame()
    else:
        first_reassign_cp = after_dead.groupby("neuron_id", sort=False)["checkpoint_idx"].min()
        fr = first_reassign_cp.rename("first_reassign_checkpoint").reset_index()
        at_first = after_dead.merge(
            fr,
            left_on=["neuron_id", "checkpoint_idx"],
            right_on=["neuron_id", "first_reassign_checkpoint"],
        )
        digits_col = (
            at_first.groupby(["neuron_id", "layer_name", "first_reassign_checkpoint"], sort=False)["digit"]
            .apply(lambda s: ",".join(map(str, sorted(s.unique()))))
            .reset_index(name="digits_at_first_reassign")
        )
        df_reassigned_summary = digits_col.merge(
            first_dead.reset_index(), on="neuron_id", how="left"
        )
        df_reassigned_summary.insert(
            0,
            "compact",
            df_reassigned_summary["neuron_id"].map(_compact_neuron_label),
        )
        df_reassigned_summary = df_reassigned_summary.sort_values(
            ["layer_name", "first_reassign_checkpoint", "compact"],
            kind="stable",
        )

    print(
        f"K_SAMPLES={_NETWORK_SAMPLES_PER_DIGIT} "
        "(assigned means all K post-nl activations > 0 for that digit).\n"
        f"Reassigned neurons: {len(reassigned_neuron_ids)}"
    )
    if not df_reassigned_summary.empty:
        display(df_reassigned_summary)
